In [37]:
import os, sys
import pandas as pd

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))

ROOT_DIR = os.path.dirname(NOTEBOOK_DIR)
SRC_DIR = os.path.join(ROOT_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
from load import load_file , CSV_PATH
df = load_file(CSV_PATH)
df.head(5)

2026-09-26 16:17:11,672 - INFO - [load.py] - loaded dataset. Shape: 168446 rows, 17 columns.


,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent
0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,4 Marla,For Sale,2,02-04-2019,NaN,NaN
1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,5.6 Marla,For Sale,3,05-04-2019,NaN,NaN
2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,8 Marla,For Sale,5,07-17-2019,NaN,NaN
3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,2 Kanal,For Sale,4,04-05-2019,NaN,NaN
4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,For Sale,3,07-10-2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168446 entries, 0 to 168445
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   property_id    168446 non-null  int64  
 1   location_id    168446 non-null  int64  
 2   page_url       168446 non-null  object 
 3   property_type  168446 non-null  object 
 4   price          168446 non-null  int64  
 5   location       168446 non-null  object 
 6   city           168446 non-null  object 
 7   province_name  168446 non-null  object 
 8   latitude       168446 non-null  float64
 9   longitude      168446 non-null  float64
 10  baths          168446 non-null  int64  
 11  area           168446 non-null  object 
 12  purpose        168446 non-null  object 
 13  bedrooms       168446 non-null  int64  
 14  date_added     168446 non-null  object 
 15  agency         124375 non-null  object 
 16  agent          124374 non-null  object 
dtypes: float64(2), int64(5), obje

In [19]:
df.columns

Index(['property_id', 'location_id', 'page_url', 'property_type', 'price',
       'location', 'city', 'province_name', 'latitude', 'longitude', 'baths',
       'area', 'purpose', 'bedrooms', 'date_added', 'agency', 'agent'],
      dtype='object')

In [30]:
# see where are the missing values
df.isnull().sum()

property_id          0
location_id          0
page_url             0
property_type        0
price                0
location             0
city                 0
province_name        0
latitude             0
longitude            0
baths                0
area                 0
purpose              0
bedrooms             0
date_added           0
agency           44071
agent            44072
area_marla           7
dtype: int64

In [41]:
from clean_data import filtered_df
flt_df = filtered_df(df)
flt_df.head()

,property_type,price,location,city,baths,area,purpose,bedrooms
0,Flat,10000000,G-10,Islamabad,2,4 Marla,For Sale,2
1,Flat,6900000,E-11,Islamabad,3,5.6 Marla,For Sale,3
2,House,16500000,G-15,Islamabad,6,8 Marla,For Sale,5
3,House,43500000,Bani Gala,Islamabad,4,2 Kanal,For Sale,4
4,House,7000000,DHA Defence,Islamabad,3,8 Marla,For Sale,3


In [39]:
flt_df['purpose'].value_counts()

purpose
For Sale    120655
For Rent     47791
Name: count, dtype: int64

In [24]:
def convert_to_marla(text_value):

    if pd.isna(text_value):
        return None
    
    text_clean = str(text_value).lower().strip()
    
    parts = text_clean.split()
    if len(parts) == 0:
        return None
        
    try:
        number = float(parts[0])
    except ValueError:
        return None

    if 'kanal' in text_clean:
        return number * 20
    else:
        return number

print("Conversion function created successfully!")
df['area_marla'] = df['area'].apply(convert_to_marla)
final_df = df.dropna()
final_df = final_df[(final_df['area_marla'] > 0) & (final_df['price'] > 100_000)]
final_df.head()

Conversion function created successfully!


,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent,area_marla
4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,For Sale,3,07-10-2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,8.0
7,1258636,3241,https://www.zameen.com/Property/e_11_e_11_4_ap...,Flat,7800000,E-11,Islamabad,Islamabad Capital,33.698244,72.984238,2,6.2 Marla,For Sale,2,05-05-2019,Ettemad Enterprises,Balqiaz Marwat,6.2
8,1402466,376,https://www.zameen.com/Property/dha_defence_dh...,House,50000000,DHA Defence,Islamabad,Islamabad Capital,33.540894,73.095732,7,1 Kanal,For Sale,7,10-19-2018,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,20.0
9,1418706,3282,https://www.zameen.com/Property/f_11_f_11_1_f_...,Penthouse,40000000,F-11,Islamabad,Islamabad Capital,33.679211,72.988787,5,1 Kanal,For Sale,5,06-27-2019,Crown Associate,Abrar Ahmed,20.0
10,1425602,429,https://www.zameen.com/Property/islamabad_dipl...,Flat,35000000,Diplomatic Enclave,Islamabad,Islamabad Capital,33.728873,73.119628,3,7.1 Marla,For Sale,3,06-03-2019,Al Sahar Estate,Zahid H. Usmani,7.1


In [4]:
X = final_df[['area_marla']]
y = final_df['price']
print("Features (X) and Target (y) created successfully!")

Features (X) and Target (y) created successfully!


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training and testing sets created successfully! from {X.size}")
print(f"Row assigned to training data {X_train.size}")
print(f"Row assigned to training data {X_test.size}")

Training and testing sets created successfully! from 87930
Row assigned to training data 70344
Row assigned to training data 17586


In [6]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
print("Model training complete!")

Model training complete!


In [7]:
# Convert values from scientific notation to human-readable PKR
base_price = model.intercept_
price_per_marla = model.coef_[0]

print(f"Base Starting Price (Intercept): {base_price:,.2f} PKR")
print(f"Value Added Per Marla (Coefficient): {price_per_marla:,.2f} PKR")


Base Starting Price (Intercept): 19,189,470.31 PKR
Value Added Per Marla (Coefficient): 914,966.65 PKR


In [8]:
# Have the model predict prices for the testing set
y_pred = model.predict(X_test)

# Show a quick comparison of the first 5 houses
comparison_df = pd.DataFrame({
    'Actual Price': y_test.head().values,
    'Predicted Price': y_pred[:5]
})
# Format numbers with commas to make them easier to read
comparison_df.style.format("{:,.2f}")


,Actual Price,Predicted Price
0,"22,000,000.00","32,913,970.00"
1,"15,600,000.00","23,764,303.54"
2,"9,500,000.00","23,764,303.54"
3,"31,500,000.00","28,339,136.77"
4,"12,500,000.00","27,424,170.13"


In [9]:
# from sklearn.metrics import r2_score
from validation import Validation
validation_obj = Validation()
# 1. Calculate the Mean Absolute Error (Average error in PKR)
mae = validation_obj.mean_absolute_error(y_test,y_pred)

# 2. Calculate the R-Squared score (Accuracy percentage)
# r2 = r2_score(y_test, y_pred)
r2 = validation_obj.r2_score(y_test,y_pred)
print("--- MODEL EXAM RESULTS ---")
print(f"Mean Absolute Error (MAE): {mae:,.2f} PKR")
print(f"R-Squared (R²) Score: {r2:.4f} (or {r2 * 100:.2f}%)")


--- MODEL EXAM RESULTS ---
Mean Absolute Error (MAE): 18,739,645.08 PKR
R-Squared (R²) Score: 0.2801 (or 28.01%)


In [10]:
from sklearn.model_selection import train_test_split
A_train, A_test, B_train, B_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
from sklearn.tree import DecisionTreeRegressor
tree_model = DecisionTreeRegressor(random_state=1)
tree_model.fit(A_train, B_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",1
,"max_le

In [12]:
B_pred = tree_model.predict(A_test) # Predicting price for a house with 5 marlas
comparision_df = pd.DataFrame({
                'Actual Price':B_test.head().values,
                'Predicted Price':B_pred[:5]
})
comparison_df.style.format("{:,.2f}")

,Actual Price,Predicted Price
0,"22,000,000.00","32,913,970.00"
1,"15,600,000.00","23,764,303.54"
2,"9,500,000.00","23,764,303.54"
3,"31,500,000.00","28,339,136.77"
4,"12,500,000.00","27,424,170.13"


In [13]:
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(B_test,B_pred)
r2 = r2_score(B_test,B_pred)
print("--- MODEL EXAM RESULTS ---")
print(f"Mean Absolute Error (MAE): {mae:,.2f} PKR")
print(f"R-Squared (R²) Score: {r2:.4f} (or {r2 * 100:.2f}%)")


--- MODEL EXAM RESULTS ---
Mean Absolute Error (MAE): 10,825,939.15 PKR
R-Squared (R²) Score: 0.5762 (or 57.62%)
